# Notebook 09: Handling Class Imbalance

## Why This Matters

Class imbalance is the norm, not the exception, in the domains where ML matters most: fraud detection (0.1% fraud), medical diagnosis (rare diseases), network intrusion detection, and churn prediction. A naive model that predicts the majority class for every sample achieves 99.9% accuracy on a 1:1000 imbalanced dataset while being completely useless. This distinction between accuracy and utility is a fundamental test of ML maturity, and every interview for applied ML roles will touch on it. The community has developed a rich toolkit — resampling, cost-sensitive learning, threshold calibration, and appropriate evaluation metrics — and understanding when to use each technique is essential.

## 1. The Imbalance Problem: Why Accuracy Fails

Consider a fraud detection problem where 1% of transactions are fraudulent. A model that always predicts "not fraud" achieves 99% accuracy — better than most models you could train — but has zero recall on the fraud class and is completely useless.

The core issue: standard model training minimizes overall loss, which is dominated by the majority class. The model learns to mostly predict the majority class because that minimizes error on the training distribution.

The fix has three components:
1. Use the right **evaluation metric** (not accuracy)
2. **Adjust the training signal** (resampling or class weights)
3. **Calibrate the decision threshold** for the specific cost structure

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler

# Create an imbalanced dataset
X, y = make_classification(
    n_samples=10000, n_features=20, n_informative=8,
    n_redundant=4, weights=[0.97, 0.03],  # 97% negative, 3% positive
    random_state=42
)
print(f"Class distribution: {np.bincount(y)}")
print(f"Minority class: {y.mean()*100:.1f}%")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [ ]:
# The accuracy trap: naive model vs. baseline model
from sklearn.dummy import DummyClassifier

# Majority-class baseline
dummy = DummyClassifier(strategy='most_frequent').fit(X_train_s, y_train)
y_dummy = dummy.predict(X_test_s)

# Standard logistic regression (no imbalance handling)
lr_naive = LogisticRegression(max_iter=500).fit(X_train_s, y_train)
y_naive = lr_naive.predict(X_test_s)
y_naive_prob = lr_naive.predict_proba(X_test_s)[:, 1]

def print_metrics(name, y_true, y_pred, y_prob=None):
    print(f"\n{name}:")
    print(f"  Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  F1:        {f1_score(y_true, y_pred):.4f}")
    if y_prob is not None:
        print(f"  AUC-ROC:   {roc_auc_score(y_true, y_prob):.4f}")
        print(f"  Avg Prec (PR-AUC): {average_precision_score(y_true, y_prob):.4f}")

print_metrics('Majority-class dummy (always predicts 0)', y_test, y_dummy)
print_metrics('Naive LR (no imbalance handling)', y_test, y_naive, y_naive_prob)

## 2. Class Weights: Cost-Sensitive Learning

The simplest fix: tell the model that misclassifying the minority class is more expensive. `class_weight='balanced'` automatically sets weights inversely proportional to class frequency:

$$w_k = \frac{n_{\text{total}}}{n_{\text{classes}} \times n_k}$$

This is equivalent to **oversampling** the minority class in the loss calculation. It is available in virtually all sklearn classifiers (`class_weight` parameter) and has zero computational overhead — no data augmentation, no extra training time.

When to use: always try this first. It is the most reliable, cheapest, and easiest to deploy in production (no data pipeline changes needed).

In [ ]:
lr_balanced = LogisticRegression(max_iter=500, class_weight='balanced').fit(X_train_s, y_train)
y_balanced = lr_balanced.predict(X_test_s)
y_balanced_prob = lr_balanced.predict_proba(X_test_s)[:, 1]

print_metrics('LR with class_weight="balanced"', y_test, y_balanced, y_balanced_prob)

# Try different weight ratios
print("\nEffect of manual class weight on F1 and Recall:")
print(f"{'weight_ratio':15s} | {'F1':>6s} | {'Recall':>8s} | {'Precision':>10s} | {'AUC':>6s}")
print('-' * 55)
for ratio in [1, 5, 10, 20, 33, 50, 100]:
    cw = {0: 1, 1: ratio}
    m = LogisticRegression(max_iter=500, class_weight=cw).fit(X_train_s, y_train)
    yp = m.predict(X_test_s)
    yp_prob = m.predict_proba(X_test_s)[:, 1]
    print(f"{str(ratio)+':1':15s} | {f1_score(y_test, yp):.4f} | {recall_score(y_test, yp):.4f}   | {precision_score(y_test, yp, zero_division=0):.6f}   | {roc_auc_score(y_test, yp_prob):.4f}")

## 3. Resampling: SMOTE and Undersampling

**Oversampling**: increase minority class samples
- **Random oversampling**: duplicate minority samples (fast; can overfit to specific minority examples)
- **SMOTE** (Synthetic Minority Oversampling Technique): generate synthetic minority samples by interpolating between existing minority samples and their k-nearest neighbors. More robust than random duplication.

**Undersampling**: reduce majority class samples
- **Random undersampling**: discard random majority samples (fast; can lose useful information)
- **Tomek Links**: remove majority samples that are near-boundary (borderline majority samples); cleans the class boundary
- **NearMiss**: keep majority samples closest to minority (preserves the hardest cases)

**Combination**: SMOTE + Tomek Links or SMOTE + ENN (Edited Nearest Neighbors) is often the strongest resampling strategy.

Critical caveat: **always apply resampling only to training data, never to test/validation data**. Use sklearn pipelines or imbalanced-learn's pipeline to enforce this.

In [ ]:
try:
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    from imblearn.under_sampling import RandomUnderSampler, TomekLinks
    from imblearn.combine import SMOTETomek
    from imblearn.pipeline import Pipeline as ImbPipeline
    HAS_IMBLEARN = True
    print("imbalanced-learn available")
except ImportError:
    HAS_IMBLEARN = False
    print("imbalanced-learn not installed. Install with: pip install imbalanced-learn")
    print("Continuing with manual SMOTE implementation.")

In [ ]:
# Manual SMOTE implementation to show the algorithm
def smote_simple(X_min, n_samples, k=5, random_state=42):
    """Simple SMOTE: generate synthetic minority samples."""
    from sklearn.neighbors import NearestNeighbors
    rng = np.random.default_rng(random_state)
    nn = NearestNeighbors(n_neighbors=k+1).fit(X_min)
    _, indices = nn.kneighbors(X_min)
    
    synthetic = []
    for _ in range(n_samples):
        i = rng.integers(0, len(X_min))
        neighbor_idx = indices[i, rng.integers(1, k+1)]  # skip self (idx 0)
        lam = rng.random()
        synthetic.append(X_min[i] + lam * (X_min[neighbor_idx] - X_min[i]))
    return np.array(synthetic)

X_min = X_train_s[y_train == 1]
X_maj = X_train_s[y_train == 0]
n_to_generate = len(X_maj) - len(X_min)

X_synthetic = smote_simple(X_min, n_to_generate)
X_resampled = np.vstack([X_train_s, X_synthetic])
y_resampled = np.hstack([y_train, np.ones(n_to_generate, dtype=int)])

print(f"Original training class distribution: {np.bincount(y_train)}")
print(f"After SMOTE:                           {np.bincount(y_resampled)}")

lr_smote = LogisticRegression(max_iter=500).fit(X_resampled, y_resampled)
y_smote = lr_smote.predict(X_test_s)
y_smote_prob = lr_smote.predict_proba(X_test_s)[:, 1]
print_metrics('LR + manual SMOTE', y_test, y_smote, y_smote_prob)

In [ ]:
if HAS_IMBLEARN:
    strategies = [
        ('SMOTE', ImbPipeline([('smote', SMOTE(random_state=42)), ('lr', LogisticRegression(max_iter=500))])),
        ('RandomOver', ImbPipeline([('ro', RandomOverSampler(random_state=42)), ('lr', LogisticRegression(max_iter=500))])),
        ('RandomUnder', ImbPipeline([('ru', RandomUnderSampler(random_state=42)), ('lr', LogisticRegression(max_iter=500))])),
        ('SMOTETomek', ImbPipeline([('st', SMOTETomek(random_state=42)), ('lr', LogisticRegression(max_iter=500))])),
    ]
    for name, pipe in strategies:
        pipe.fit(X_train_s, y_train)
        yp = pipe.predict(X_test_s)
        yp_prob = pipe.predict_proba(X_test_s)[:, 1]
        print(f"{name:15s} | F1={f1_score(y_test, yp):.4f}  Recall={recall_score(y_test, yp):.4f}  PR-AUC={average_precision_score(y_test, yp_prob):.4f}")

## 4. Threshold Tuning

By default, classifiers predict the positive class when predicted probability ≥ 0.5. For imbalanced classes, this threshold is often wrong. The optimal threshold depends on the **cost structure** of your problem:

- **High recall needed** (fraud, medical): lower the threshold → catch more positives, but more false alarms
- **High precision needed** (spam email): raise the threshold → fewer false positives, but miss some positives

Two tools to find the optimal threshold:
1. **ROC curve**: plot TPR vs. FPR at all thresholds; choose the point closest to (0, 1) or where F1 is maximized
2. **Precision-Recall curve**: plot precision vs. recall at all thresholds; AUC-PR is a better metric than AUC-ROC for imbalanced classes because it focuses on the minority class

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

# Use balanced LR probabilities
y_prob = y_balanced_prob  # from class_weight='balanced'

fpr, tpr, roc_thresholds = roc_curve(y_test, y_prob)
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_prob)

# F1 at each threshold
f1_at_thresh = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-10)
best_thresh_idx = np.argmax(f1_at_thresh)
best_thresh = pr_thresholds[best_thresh_idx]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(fpr, tpr, 'steelblue', linewidth=2)
axes[0].plot([0,1],[0,1],'k--', alpha=0.5)
axes[0].set_xlabel('FPR')
axes[0].set_ylabel('TPR')
axes[0].set_title(f'ROC Curve (AUC={roc_auc_score(y_test, y_prob):.4f})')

axes[1].plot(recall, precision, 'coral', linewidth=2)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'PR Curve (AUC={average_precision_score(y_test, y_prob):.4f})')
axes[1].scatter(recall[best_thresh_idx], precision[best_thresh_idx], s=100, c='black', zorder=5,
                label=f'Best thresh={best_thresh:.3f}')
axes[1].legend()

axes[2].plot(pr_thresholds, f1_at_thresh, 'purple', linewidth=2)
axes[2].axvline(best_thresh, color='red', linestyle='--', label=f'Best F1 thresh={best_thresh:.3f}')
axes[2].set_xlabel('Threshold')
axes[2].set_ylabel('F1 Score')
axes[2].set_title('F1 Score vs. Decision Threshold')
axes[2].legend()

plt.tight_layout()
plt.show()

y_tuned = (y_prob >= best_thresh).astype(int)
print_metrics(f'LR balanced + threshold={best_thresh:.3f}', y_test, y_tuned, y_prob)

## 5. Evaluation Metrics for Imbalanced Classes

Choosing the right metric is as important as choosing the right model. The metric must reflect the actual cost structure of the problem.

| Metric | Formula | Use When |
|--------|---------|----------|
| Accuracy | (TP+TN)/(P+N) | Balanced classes only |
| Precision | TP/(TP+FP) | Cost of false positives is high (spam, alerts) |
| Recall / Sensitivity | TP/(TP+FN) | Cost of false negatives is high (fraud, diagnosis) |
| F1 Score | 2PR/(P+R) | Need balance of precision and recall |
| F-beta | (1+β²)PR/((β²P)+R) | β>1 weights recall; β<1 weights precision |
| AUC-ROC | Area under ROC | Ranking quality; invariant to threshold |
| AUC-PR | Area under PR curve | **Best for imbalanced classes** — focuses on minority class |

In [ ]:
# Comprehensive comparison: all strategies
from sklearn.metrics import fbeta_score

results = []

strategies_eval = [
    ('Majority-class dummy', dummy.predict(X_test_s), np.zeros(len(y_test))),
    ('LR (no handling)', y_naive, y_naive_prob),
    ('LR + class_weight balanced', y_balanced, y_balanced_prob),
    ('LR + SMOTE (manual)', y_smote, y_smote_prob),
    (f'LR balanced + threshold={best_thresh:.2f}', y_tuned, y_prob),
]

print(f"{'Strategy':35s} | {'Acc':6s} | {'Prec':6s} | {'Rec':6s} | {'F1':6s} | {'AUC-ROC':8s} | {'PR-AUC':6s}")
print('-' * 95)
for name, yp, yp_prob in strategies_eval:
    acc  = accuracy_score(y_test, yp)
    prec = precision_score(y_test, yp, zero_division=0)
    rec  = recall_score(y_test, yp)
    f1   = f1_score(y_test, yp)
    if np.any(yp_prob != 0):
        auc = roc_auc_score(y_test, yp_prob)
        prauc = average_precision_score(y_test, yp_prob)
    else:
        auc = prauc = float('nan')
    print(f"{name:35s} | {acc:6.4f} | {prec:6.4f} | {rec:6.4f} | {f1:6.4f} | {auc:8.4f} | {prauc:6.4f}")

## Summary

| Strategy | Cost | Deployment Complexity | Best For |
|----------|------|----------------------|----------|
| class_weight='balanced' | Zero extra cost | Zero (just a parameter) | Always try first |
| Threshold tuning | Zero extra cost | Low (adjust at inference) | When you have a specific precision/recall target |
| Random oversampling | Low | Low | Simple baseline resampling |
| Random undersampling | Low | Low | When majority class is huge and noisy |
| SMOTE | Moderate | Moderate (offline preprocessing) | When more minority samples needed; avoid at 1:1000+ ratios |
| SMOTETomek / SMOTE+ENN | Higher | Moderate | Highest quality resampling |

| Metric | Why It Matters |
|--------|----------------|
| PR-AUC | Better than AUC-ROC for imbalanced classes; focuses on minority class performance |
| F1 / F-beta | Single number combining precision and recall; β controls trade-off |
| Recall | Most important when missing a positive is catastrophic (fraud, disease) |